# Pipeline Reset & Backfill
This notebook resets the transaction pipeline state and re-runs notebooks 03 → 04b → 05
to backfill columns fixed in the transform:
- **TRANSACTION_CATEGORY** and **TRANSACTION_DESCRIPTION** (were missing from outcols)
- **AC_ENTRY_SR_NO** (was cast as int, overflowed for 8B+ values — now long/bigint)
- **RELATED_ACCOUNT** and **RELATED_CUSTOMER** (`.0` suffix stripped)

⚠️ **Run cells in order.** Cell 4 is destructive (truncates tables + resets watermark).

In [0]:
# Step 1: Truncate curated + mart tables, reset watermark, clear affected_dates
# This is REQUIRED because AC_ENTRY_SR_NO is part of the TRANSACTION_KEY hash.
# Old keys used NULL (int overflow); new keys will use the real value (long).
# Without truncation, the MERGE would create duplicate rows.

tables_to_truncate = [
    "ng_banking_lakehouse.transaction_curated.transaction_entries",
    "ng_banking_lakehouse.transaction_curated.pipeline_affected_dates",
    "ng_banking_lakehouse.transaction_mart.daily_transaction_activity",
    "ng_banking_lakehouse.transaction_mart.account_transaction_activity",
    "ng_banking_lakehouse.transaction_mart.branch_transaction_activity",
    "ng_banking_lakehouse.transaction_mart.transaction_code_activity",
    "ng_banking_lakehouse.transaction_mart.transaction_category_activity",
    "ng_banking_lakehouse.transaction_mart.aml_exception_activity",
]

for tbl in tables_to_truncate:
    spark.sql(f"TRUNCATE TABLE {tbl}")
    print(f"Truncated: {tbl}")

# Reset watermark so notebook 03 re-processes all raw data
spark.sql("""
    MERGE INTO ng_banking_lakehouse.transaction_curated.etl_watermark t
    USING (SELECT 'ng_banking_transaction_lakehouse_pipeline' PIPELINE_NAME,
                  TIMESTAMP('1900-01-01 00:00:00') LAST_RAW_INGEST_TIMESTAMP,
                  current_timestamp() LAST_RUN_TIMESTAMP) s
    ON t.PIPELINE_NAME = s.PIPELINE_NAME
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
# Remove any stale watermark rows with incorrect pipeline name from a previous run
spark.sql("DELETE FROM ng_banking_lakehouse.transaction_curated.etl_watermark WHERE PIPELINE_NAME = 'ng_banking_transaction_lakehouse'")
print("\nWatermark reset to 1900-01-01")
print("Ready to re-run pipeline.")

In [0]:
# Step 2: Re-run notebook 03 (incremental transform) — repopulates curated table
# with fixed TRANSACTION_CATEGORY, TRANSACTION_DESCRIPTION, AC_ENTRY_SR_NO (long),
# and cleaned RELATED_ACCOUNT/RELATED_CUSTOMER (no .0 suffix).
result_03 = dbutils.notebook.run(
    "/Users/warubayi@yahoo.com/ng_banking_transaction_lakehouse_pipeline/notebooks/03_transaction_incremental_transform",
    1200,
    {"catalog": "ng_banking_lakehouse", "pipeline_name": "ng_banking_transaction_lakehouse_pipeline", "environment": "dev"}
)
print(f"Notebook 03 result: {result_03}")

In [0]:
# Step 3: Re-run notebook 04b (quality gate) — validates the refreshed curated data
result_04b = dbutils.notebook.run(
    "/Users/warubayi@yahoo.com/ng_banking_transaction_lakehouse_pipeline/notebooks/04b_transaction_quality_gate",
    600,
    {"catalog": "ng_banking_lakehouse", "pipeline_name": "ng_banking_transaction_lakehouse_pipeline", "environment": "dev"}
)
print(f"Notebook 04b result: {result_04b}")

In [0]:
# Step 4: Re-run notebook 05 (marts) — rebuilds all 6 mart tables with fixed data
result_05 = dbutils.notebook.run(
    "/Users/warubayi@yahoo.com/ng_banking_transaction_lakehouse_pipeline/notebooks/05_transaction_mart",
    1200,
    {"catalog": "ng_banking_lakehouse", "pipeline_name": "ng_banking_transaction_lakehouse_pipeline", "environment": "dev"}
)
print(f"Notebook 05 result: {result_05}")

In [0]:
# Step 5: Verify the fixes
from pyspark.sql import functions as F

curated = spark.table("ng_banking_lakehouse.transaction_curated.transaction_entries")
total = curated.count()

print(f"=== CURATED TABLE: {total} rows ===\n")
check_cols = [
    'TRANSACTION_CATEGORY', 'TRANSACTION_DESCRIPTION', 'AC_ENTRY_SR_NO',
    'RELATED_ACCOUNT', 'RELATED_CUSTOMER', 'AML_EXCEPTION'
]
for col_name in check_cols:
    non_null = curated.filter(F.col(col_name).isNotNull()).count()
    samples = curated.filter(F.col(col_name).isNotNull()).select(col_name).distinct().limit(5).collect()
    sample_vals = [str(r[col_name]) for r in samples]
    print(f"{col_name:<30} {non_null:>6}/{total} non-null  samples: {sample_vals}")

print("\n=== MART TABLES ===")
for name, tbl in [
    ("transaction_code_activity", "ng_banking_lakehouse.transaction_mart.transaction_code_activity"),
    ("transaction_category_activity", "ng_banking_lakehouse.transaction_mart.transaction_category_activity"),
    ("account_transaction_activity", "ng_banking_lakehouse.transaction_mart.account_transaction_activity"),
]:
    df = spark.table(tbl)
    cnt = df.count()
    if 'TRANSACTION_CATEGORY' in df.columns:
        tc_nn = df.filter(F.col('TRANSACTION_CATEGORY').isNotNull()).count()
        print(f"{name:<35} {cnt} rows, TRANSACTION_CATEGORY non-null: {tc_nn}")
    elif 'UNIQUE_TRANSACTION_CATEGORIES' in df.columns:
        utc_pos = df.filter(F.col('UNIQUE_TRANSACTION_CATEGORIES') > 0).count()
        print(f"{name:<35} {cnt} rows, UNIQUE_TRANSACTION_CATEGORIES > 0: {utc_pos}")
    if 'RELATED_CUSTOMER' in df.columns:
        rc_nn = df.filter(F.col('RELATED_CUSTOMER').isNotNull()).count()
        samples = df.filter(F.col('RELATED_CUSTOMER').isNotNull()).select('RELATED_CUSTOMER').distinct().limit(3).collect()
        print(f"  {'':33} RELATED_CUSTOMER: {rc_nn} non-null, samples: {[str(r[0]) for r in samples]}")
